# Dataloading in Practice

Many “slow model” complaints are really **slow input pipeline** complaints. This notebook focuses on PyTorch's native `Dataset` and `DataLoader` tools and shows how a few configuration choices change throughput.

We will look at five practical ideas:

1. `num_workers` for parallel data preparation
2. `pin_memory` for faster host-to-GPU transfer when CUDA is available
3. `prefetch_factor` and `persistent_workers` to reduce startup overhead
4. Simple **sharding** for iterable datasets so worker processes do not duplicate work
5. Overlapping CPU dataloading with GPU model execution so the next batch is prepared while the current batch is still training


In [1]:
import platform
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, IterableDataset, get_worker_info

torch.manual_seed(0)
np.random.seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Platform: {platform.system()}')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


CUDA available: True
Platform: Linux
Device: cuda
GPU: NVIDIA RTX 6000 Ada Generation


In [2]:
class SlowFieldDataset(Dataset):
    # Simulates gridded data that needs I/O and CPU-side preprocessing.
    def __init__(self, n_samples=4096, size=64, io_delay=0.002, transform_repeats=3):
        self.n_samples = n_samples
        self.size = size
        self.io_delay = io_delay
        self.transform_repeats = transform_repeats
        x = np.linspace(-1.0, 1.0, size, dtype=np.float32)
        self.xx, self.yy = np.meshgrid(x, x)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        rng = np.random.default_rng(idx)
        field = np.sin((idx % 17 + 1) * self.xx) + np.cos((idx % 11 + 1) * self.yy)
        field = field + rng.normal(scale=0.05, size=field.shape)
        field = field.astype(np.float32)

        time.sleep(self.io_delay)

        for _ in range(self.transform_repeats):
            field = 0.6 * field + 0.25 * np.roll(field, 1, axis=0) + 0.15 * np.roll(field, -1, axis=1)
        target = np.int64(field.mean() > 0.05)
        return torch.from_numpy(field[None, ...]), torch.tensor(target)


@dataclass
class LoaderBenchmark:
    config: str
    batches: int
    epoch_seconds: float
    batches_per_second: float
    samples_per_second: float


In [3]:
def make_loader(dataset, *, batch_size, num_workers, pin_memory, prefetch_factor=None, persistent_workers=False, shuffle=False):
    kwargs = dict(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    if num_workers > 0:
        kwargs['persistent_workers'] = persistent_workers
        if prefetch_factor is not None:
            kwargs['prefetch_factor'] = prefetch_factor
        if platform.system() != 'Windows':
            kwargs['multiprocessing_context'] = 'fork'
    return DataLoader(**kwargs)


def benchmark_loader(name, loader, max_batches=40):
    start = time.perf_counter()
    batches = 0
    samples = 0
    for batches, (xb, yb) in enumerate(loader, start=1):
        samples += xb.size(0)
        if batches >= max_batches:
            break
    elapsed = time.perf_counter() - start
    return LoaderBenchmark(
        config=name,
        batches=batches,
        epoch_seconds=elapsed,
        batches_per_second=batches / elapsed,
        samples_per_second=samples / elapsed,
    )


## Worker Parallelism and Pipeline Settings

The dataset below is purposely slow. Each item sleeps briefly to imitate file I/O, then does a few CPU operations to imitate preprocessing. This is exactly the situation where `num_workers` and prefetching matter.


In [4]:
dataset = SlowFieldDataset()
configs = {
    'workers_0': dict(batch_size=32, num_workers=0, pin_memory=False, prefetch_factor=None, persistent_workers=False),
    'workers_2': dict(batch_size=32, num_workers=2, pin_memory=False, prefetch_factor=2, persistent_workers=True),
    'workers_4': dict(batch_size=32, num_workers=4, pin_memory=False, prefetch_factor=2, persistent_workers=True),
    'workers_4_pinned': dict(batch_size=32, num_workers=4, pin_memory=torch.cuda.is_available(), prefetch_factor=2, persistent_workers=True),
}

rows = []
for name, cfg in configs.items():
    loader = make_loader(dataset, **cfg)
    rows.append(benchmark_loader(name, loader).__dict__)

pd.DataFrame(rows).sort_values('samples_per_second', ascending=False)


,config,batches,epoch_seconds,batches_per_second,samples_per_second
2,workers_4,40,0.838918,47.680484,1525.775490
3,workers_4_pinned,40,0.896686,44.608726,1427.479223
1,workers_2,40,1.597577,25.037924,801.213570
0,workers_0,40,3.077997,12.995466,415.854907


A few rules usually emerge:

- Increasing `num_workers` helps when examples are slow to prepare.
- More workers is not always better; too many processes can fight over CPU and memory.
- `pin_memory=True` matters only if you are feeding a CUDA device.
- `persistent_workers=True` helps when you iterate over the same dataloader for many epochs.


In [5]:
warm_loader = make_loader(dataset, batch_size=64, num_workers=4, pin_memory=torch.cuda.is_available(), prefetch_factor=2, persistent_workers=True)

first_pass = benchmark_loader('first_pass', warm_loader, max_batches=30)
second_pass = benchmark_loader('second_pass', warm_loader, max_batches=30)
pd.DataFrame([first_pass.__dict__, second_pass.__dict__])


,config,batches,epoch_seconds,batches_per_second,samples_per_second
0,first_pass,30,1.283364,23.376071,1496.068513
1,second_pass,30,1.548488,19.373735,1239.919015


## Add a Dummy Model So Transfers Matter

Loader-only timing is useful, but `pin_memory` is really about *feeding a GPU efficiently*. To make that visible we will attach a small convolutional model and measure a few training steps.

The setup below uses `non_blocking=True` when moving tensors to CUDA. That flag is what lets pinned host memory help: page-locked memory can be copied to the GPU asynchronously, while pageable memory generally cannot.


In [6]:
class TinyStormCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.net(x)


@dataclass
class TrainBenchmark:
    config: str
    steps: int
    total_seconds: float
    transfer_seconds: float
    compute_seconds: float
    samples_per_second: float


def benchmark_training(name, loader, steps=25):
    model = TinyStormCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    total_examples = 0
    transfer_seconds = 0.0
    compute_seconds = 0.0
    start = time.perf_counter()

    for step, (xb, yb) in enumerate(loader, start=1):
        t0 = time.perf_counter()
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        transfer_seconds += time.perf_counter() - t0

        t1 = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        if device.type == 'cuda':
            torch.cuda.synchronize()
        compute_seconds += time.perf_counter() - t1

        total_examples += xb.size(0)
        if step >= steps:
            break

    total_seconds = time.perf_counter() - start
    return TrainBenchmark(
        config=name,
        steps=step,
        total_seconds=total_seconds,
        transfer_seconds=transfer_seconds,
        compute_seconds=compute_seconds,
        samples_per_second=total_examples / total_seconds,
    )


In [7]:
train_dataset = SlowFieldDataset(n_samples=4096, size=64, io_delay=0.0015, transform_repeats=4)

train_configs = {
    'no_pin': dict(batch_size=64, num_workers=4, pin_memory=False, prefetch_factor=2, persistent_workers=True, shuffle=True),
    'pin_memory': dict(batch_size=64, num_workers=4, pin_memory=torch.cuda.is_available(), prefetch_factor=2, persistent_workers=True, shuffle=True),
}

train_rows = []
for name, cfg in train_configs.items():
    loader = make_loader(train_dataset, **cfg)
    train_rows.append(benchmark_training(name, loader).__dict__)

pd.DataFrame(train_rows)


,config,steps,total_seconds,transfer_seconds,compute_seconds,samples_per_second
0,no_pin,25,1.094596,0.011543,0.443283,1461.726898
1,pin_memory,25,0.977996,0.007861,0.059299,1635.998338


On a CPU-only machine, the two rows above should look nearly identical because there is no host-to-GPU copy to accelerate. On CUDA, the `pin_memory` configuration often reduces the time spent in the transfer section and increases overall throughput.


## Overlap CPU Dataloading With GPU Compute

The next idea is more subtle. Even with worker processes, a plain training loop usually does this in sequence:

1. wait for the next batch to arrive from the dataloader
2. copy that batch to the GPU
3. run the model

On CUDA we can do better by asking the CPU and dataloader workers to prepare the *next* batch while the GPU is still training on the *current* one. A small prefetcher object is enough to demonstrate the pattern.


In [8]:
class CUDAPrefetcher:
    def __init__(self, loader, device):
        self.loader = iter(loader)
        self.device = device
        self.stream = torch.cuda.Stream(device=device)
        self.next_batch = None
        self.preload()

    def preload(self):
        try:
            xb, yb = next(self.loader)
        except StopIteration:
            self.next_batch = None
            return

        with torch.cuda.stream(self.stream):
            xb = xb.to(self.device, non_blocking=True)
            yb = yb.to(self.device, non_blocking=True)
        self.next_batch = (xb, yb)

    def __iter__(self):
        return self

    def __next__(self):
        if self.next_batch is None:
            raise StopIteration
        torch.cuda.current_stream(self.device).wait_stream(self.stream)
        batch = self.next_batch
        self.preload()
        return batch


def benchmark_overlap(loader, steps=30):
    model = TinyStormCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    def run(iterator):
        total_examples = 0
        start = time.perf_counter()
        for step, (xb, yb) in enumerate(iterator, start=1):
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_examples += xb.size(0)
            if step >= steps:
                break
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        return elapsed, total_examples / elapsed

    plain_iterator = ((xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)) for xb, yb in loader)
    plain_seconds, plain_throughput = run(plain_iterator)

    prefetch_seconds, prefetch_throughput = run(CUDAPrefetcher(loader, device))
    return pd.DataFrame([
        {'mode': 'plain_loop', 'seconds': plain_seconds, 'samples_per_second': plain_throughput},
        {'mode': 'cuda_prefetcher', 'seconds': prefetch_seconds, 'samples_per_second': prefetch_throughput},
    ])


In [11]:
overlap_loader = make_loader(
    train_dataset,
    batch_size=64,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
    prefetch_factor=2,
    persistent_workers=True,
    shuffle=True,
)

if device.type == 'cuda':
    result = benchmark_overlap(overlap_loader, steps=30)
else:
    result = None
    print('Overlap demo requires CUDA. The key idea is that worker processes and the main CPU thread can stage the next pinned batch while the GPU is still executing kernels on the current one.')

result


,mode,seconds,samples_per_second
0,plain_loop,1.007997,1904.767083
1,cuda_prefetcher,0.859350,2234.248084


The `CUDAPrefetcher` does not make the model itself faster. It shortens the idle gaps *between* model steps by moving the next batch onto the device in a separate CUDA stream while the default stream is busy computing on the current batch.

That is the concrete answer to “how can the CPU load the next batch while the model runs on CUDA?”

- dataloader workers keep reading and preprocessing on the CPU
- pinned host memory makes asynchronous host-to-device copies possible
- a separate CUDA stream starts the copy for the next batch early
- the training loop waits on that stream only when it actually needs the next batch


## Sharding an Iterable Dataset

Map-style datasets (`__getitem__`, `__len__`) are common, but some workflows stream data from a source and naturally fit an `IterableDataset`. In that case you must shard the stream yourself so each worker gets a disjoint subset.


In [12]:
class StreamingStormDataset(IterableDataset):
    def __init__(self, n_items=24):
        self.n_items = n_items

    def __iter__(self):
        info = get_worker_info()
        if info is None:
            worker_id = 0
            n_workers = 1
        else:
            worker_id = info.id
            n_workers = info.num_workers

        for idx in range(worker_id, self.n_items, n_workers):
            yield {'worker_id': worker_id, 'sample_id': idx}


stream_loader_kwargs = dict(batch_size=None, num_workers=3)
if platform.system() != 'Windows':
    stream_loader_kwargs['multiprocessing_context'] = 'fork'

stream_loader = DataLoader(StreamingStormDataset(n_items=16), **stream_loader_kwargs)
items = list(stream_loader)
pd.DataFrame(items)


,worker_id,sample_id
0,0,0
1,1,1
2,2,2
3,0,3
4,1,4
5,2,5
6,0,6
7,1,7
8,2,8
9,0,9


Every worker should report a different `sample_id` pattern. If multiple workers return the same IDs, your iterable dataset is duplicating examples rather than parallelizing them.

**Checklist when a dataloader is slow**

1. Time the dataloader by itself before blaming the model.
2. Try `num_workers=0, 2, 4, 8` rather than assuming.
3. Use `pin_memory=True` only when a GPU is actually consuming the batches.
4. Pair pinned memory with `non_blocking=True` on `.to(device)`.
5. For streams, shard explicitly with `get_worker_info()`.
6. If CUDA is underutilized, check whether you can overlap transfer of batch `t+1` with compute on batch `t`.
